In [ ]:
from pathlib import Path
from scipy.io import loadmat
import scipy.signal as sgn
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score, average_precision_score


In [ ]:
from Functions import remove_baseline_filter,  preprocess_one_file , stem_to_subject_id , prepare_split_data_loaders,training_curve_plot , WindowDataset, CNN1D, train_model_with_loaders, eval_loader_binary , run_one_experiment , select_channels

In [ ]:

folder = Path(r"C:\Users\saber\thesis_second_try\data\Rest\Respiration_ECG_Raw")


# Parameters
FS_IN = 1000
FS_OUT = 60
WIN_LEN = 2048
STRIDE = WIN_LEN  # non-overlap
batch_size = 64

SIGNAL_ORDER = ["ECG_mV", "Fingerpulse", "Cannula", "Thermopod"]

SCALE = {
    "ECG_mV": 1000.0,      # V -> mV
    "Fingerpulse": 10.0,
    "Cannula": 10.0,
    "Thermopod": 1.0
}

# ────────────────────────────────────────────────
# The baseline function 
SOS = remove_baseline_filter(FS_IN)

#────────────────────────────────────────────────
# Run over alla the files
mat_files = sorted(folder.glob("*.mat"))
participant_files = [f for f in mat_files if f.name.startswith("resting_state_participant_")]
print("Total .mat files:", len(mat_files))
print("Participant files:", len(participant_files))

#────────────────────────────────────────────────
# Split the files in mouth and nose 
nose_files, mouth_files, skipped = [], [], []
for f in participant_files:
    m = re.search(r"participant_(\d+)", f.stem)
    if not m:
        skipped.append(f.name)
        continue
    pid = int(m.group(1))
    if 10 <= pid <= 112:
        nose_files.append(f)
    elif 200 <= pid <= 312:
        mouth_files.append(f)
    else:
        skipped.append(f.name)

print("Nose files :", len(nose_files))
print("Mouth files:", len(mouth_files))
print("Skipped    :", len(skipped))

#────────────────────────────────────────────────
# Preprocess each group 
nose_tensors_by_participant = {}
mouth_tensors_by_participant = {}
failed = []
# 1. for nose 
for f in nose_files:
    try:
        nose_tensors_by_participant[f.stem] = preprocess_one_file(
            f,
            fs_in=FS_IN,
            fs_out=FS_OUT,
            win_len=WIN_LEN,
            stride=STRIDE,
            keep_signals=SIGNAL_ORDER,
            scale=SCALE,
            sos=SOS,
        )
    except Exception as e:
        failed.append((f.name, "nose", str(e)))
# 2. for mouth
for f in mouth_files:
    try:
        mouth_tensors_by_participant[f.stem] = preprocess_one_file(
            f,
            fs_in=FS_IN,
            fs_out=FS_OUT,
            win_len=WIN_LEN,
            stride=STRIDE,
            keep_signals=SIGNAL_ORDER,
            scale=SCALE,
            sos=SOS,
        )
    except Exception as e:
        failed.append((f.name, "mouth", str(e)))

print("Processed nose :", len(nose_tensors_by_participant))
print("Processed mouth:", len(mouth_tensors_by_participant))
print("Failed:", len(failed))

#────────────────────────────────────────────────
# Stacking the tensors 
if len(nose_tensors_by_participant) > 0:
    nose_keys = sorted(nose_tensors_by_participant.keys())
    X_nose = np.stack([nose_tensors_by_participant[k] for k in nose_keys], axis=0)
    print("X_nose shape:", X_nose.shape)  # (P_nose, n_windows, 2, 2048)

if len(mouth_tensors_by_participant) > 0:
    mouth_keys = sorted(mouth_tensors_by_participant.keys())
    X_mouth = np.stack([mouth_tensors_by_participant[k] for k in mouth_keys], axis=0)
    print("X_mouth shape:", X_mouth.shape)  # (P_mouth, n_windows, 2, 2048)


Total .mat files: 207
Participant files: 205
Nose files : 102
Mouth files: 103
Skipped    : 0
Processed nose : 102
Processed mouth: 103
Failed: 0
X_nose shape: (102, 9, 4, 2048)
X_mouth shape: (103, 9, 4, 2048)


In [ ]:
# Checking that everything is working
nose_keys = sorted(nose_tensors_by_participant.keys()) 
# print(mouth_keys)
print("X_nose shape:", X_nose .shape)
X_nose = np.stack([nose_tensors_by_participant[k] for k in nose_keys], axis=0)
print("First 5 keys:", nose_keys[:5])

X_nose shape: (102, 9, 4, 2048)
First 5 keys: ['resting_state_participant_010', 'resting_state_participant_011', 'resting_state_participant_012', 'resting_state_participant_013', 'resting_state_participant_014']


In [ ]:
import pandas as pd
Path_subject = r"df_nose.csv"
df_nose= pd.read_csv(Path_subject)
df_nose["Age_group"] = pd.qcut(df_nose["Age"], q=5, labels=False)
print(df_nose["Age_group"].value_counts())
df_nose.head()

Age_group
2    23
0    23
1    20
4    18
3    18
Name: count, dtype: int64


,Subject Number,Gender,Age,Age_group
0,10,1,37,4
1,11,1,29,2
2,12,0,25,1
3,13,0,31,3
4,14,0,28,2


In [ ]:
age_by_subject = dict(zip(df_nose["Subject Number"], df_nose["Age_group"]))
age_by_subject # this is subject id and their correct gender 


{10: 4,
 11: 2,
 12: 1,
 13: 3,
 14: 2,
 15: 1,
 16: 1,
 17: 0,
 18: 1,
 19: 1,
 20: 2,
 21: 1,
 22: 2,
 23: 3,
 24: 1,
 25: 3,
 26: 0,
 27: 2,
 28: 1,
 29: 0,
 30: 0,
 31: 0,
 33: 2,
 34: 2,
 35: 0,
 36: 1,
 37: 4,
 38: 1,
 39: 4,
 40: 3,
 41: 3,
 42: 0,
 43: 4,
 44: 0,
 45: 2,
 46: 2,
 47: 4,
 48: 2,
 49: 0,
 50: 1,
 51: 0,
 52: 0,
 53: 2,
 54: 4,
 55: 4,
 56: 4,
 57: 3,
 58: 1,
 59: 2,
 60: 3,
 61: 4,
 62: 3,
 63: 2,
 64: 2,
 65: 2,
 66: 1,
 67: 2,
 68: 3,
 69: 0,
 70: 1,
 71: 0,
 72: 0,
 73: 3,
 74: 0,
 75: 4,
 76: 2,
 77: 4,
 78: 3,
 79: 1,
 80: 1,
 81: 2,
 82: 1,
 83: 0,
 84: 0,
 85: 4,
 86: 4,
 87: 0,
 88: 4,
 89: 4,
 90: 3,
 91: 3,
 92: 2,
 93: 0,
 94: 4,
 95: 1,
 96: 3,
 97: 0,
 98: 3,
 99: 3,
 100: 3,
 101: 2,
 102: 4,
 103: 0,
 104: 1,
 105: 3,
 106: 2,
 107: 0,
 108: 1,
 109: 0,
 110: 4,
 111: 2,
 112: 2}

In [ ]:
nose_ids_from_files = np.array([stem_to_subject_id(k) for k in nose_keys], dtype=int)

print("First 10 subject IDs from files:", nose_ids_from_files[:10])
print("Number of participants from files:", len(nose_ids_from_files))
print("Unique IDs:", len(np.unique(nose_ids_from_files)))


# nose_ids_for_labels = nose_ids_from_files - 200

print("First 10 raw mouth IDs:   ", nose_ids_from_files[:10])
# print("First 10 mapped label IDs:", nose_ids_for_labels[:10])

y_nose = np.array([age_by_subject[sid] for sid in nose_ids_from_files], dtype=int)
print("y_nose age:", y_nose)
print("y_nose shape:", y_nose.shape)


First 10 subject IDs from files: [10 11 12 13 14 15 16 17 18 19]
Number of participants from files: 102
Unique IDs: 102
First 10 raw mouth IDs:    [10 11 12 13 14 15 16 17 18 19]
y_nose age: [4 2 1 3 2 1 1 0 1 1 2 1 2 3 1 3 0 2 1 0 0 0 2 2 0 1 4 1 4 3 3 0 4 0 2 2 4
 2 0 1 0 0 2 4 4 4 3 1 2 3 4 3 2 2 2 1 2 3 0 1 0 0 3 0 4 2 4 3 1 1 2 1 0 0
 4 4 0 4 4 3 3 2 0 4 1 3 0 3 3 3 2 4 0 1 3 2 0 1 0 4 2 2]
y_nose shape: (102,)


In [ ]:


train_loader, val_loader, test_loader, meta = prepare_split_data_loaders(
    X_nose, y_nose,
    seed=6,
    batch_size=batch_size,
    normalize=True,
    return_meta=True
)

# baseline: majority-class accuracy on validation participants
X_val_p = meta["X_val_p"]
y_val_p = meta["y_val_p"]
baseline_acc = max(np.mean(y_val_p == 0), np.mean(y_val_p == 1))
# ────────────────────────────────────────────────
#  Quick sanity checks on test set 

print("\nFirst 20 test window labels:", meta['y_test'][10:40])

print("Example test window [0] – channel means:", meta['X_test'][0].mean(axis=1))

# If you want to see beginning of first channel of first test window:
print("Example test window[0] channel 0 – first 30 values:")
print(meta['X_test'][0, 0, :30])

X_mouth:     (102, 9, 4, 2048)
y_mouth:     (102,)   unique: [0 1 2 3 4]   counts: [23 20 23 18 18]



After normalization (train mean per ch): [5.1315712e-09 1.7211682e-09 4.3494058e-10 1.1644766e-08]
After normalization (train std  per ch): [1.0000001 0.9999999 0.9999994 1.0000002]

Window-level shapes:
X_train:     (576, 4, 2048)  y_train: (576,)  counts: [126 117 126  99 108]
X_val  :     (153, 4, 2048)  y_val  : (153,)  counts: [36 27 36 27 27]
X_test :     (189, 4, 2048)  y_test : (189,)  counts: [45 36 45 36 27]

First 20 test window labels: [2 2 2 2 2 2 2 2 0 0 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 1 1 1 1]
Example test window [0] – channel means: [ 0.00057823 -0.00222666 -0.00129117  0.00334323]
Example test window[0] channel 0 – first 30 values:
[-0.21939328 -0.1992381   0.05615242 -0.51319504 -0.16294444 -0.26717377
 -0.08923652 -0.2950932   0.00329522 -0.09957869  0.27944967  0.37120533
 -0.3075759  -0.4381697  -0.5389357  -0.13390262 -0.35230437  2.9527228
  1.6764307  -2.4667182  -0.06636624 -0.15353942  0.15083319 -0.02865219
  0.43714947  0.24856693  0.42332456  0.9134305   1

In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def train_model_with_loaders_multiclass(
    model, train_loader, val_loader,
    n_classes=5,
    epochs=50, lr=1e-3, patience=7,
    class_weights=None,   # optional torch tensor of shape [n_classes]
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    if class_weights is not None:
        class_weights = class_weights.to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    train_probs, val_probs = [], []  # store probs per epoch (optional)

    best_val_loss = float("inf")
    best_state = None
    bad_epochs = 0

    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        # ---- train ----
        model.train()
        loss_sum, n = 0.0, 0
        y_true_tr, y_pred_tr = [], []
        epoch_train_probs = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device).long().view(-1)   # labels 0..4

            optimizer.zero_grad()
            logits = model(xb)                   # [B, 5]
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * xb.size(0)
            n += xb.size(0)

            preds = logits.argmax(dim=1)
            y_true_tr.append(yb.detach().cpu().numpy())
            y_pred_tr.append(preds.detach().cpu().numpy())

            probs = torch.softmax(logits, dim=1).detach().cpu()   # [B, 5]
            epoch_train_probs.append(probs)

        train_loss = loss_sum / n
        y_true_tr = np.concatenate(y_true_tr)
        y_pred_tr = np.concatenate(y_pred_tr)
        train_acc = accuracy_score(y_true_tr, y_pred_tr)

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        train_probs.append(torch.cat(epoch_train_probs, dim=0))   # [N_train, 5]

        # ---- val ----
        model.eval()
        loss_sum, n = 0.0, 0
        y_true_va, y_pred_va = [], []
        epoch_val_probs = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device).long().view(-1)

                logits = model(xb)
                loss = criterion(logits, yb)

                loss_sum += loss.item() * xb.size(0)
                n += xb.size(0)

                preds = logits.argmax(dim=1)
                y_true_va.append(yb.cpu().numpy())
                y_pred_va.append(preds.cpu().numpy())

                probs = torch.softmax(logits, dim=1).cpu()
                epoch_val_probs.append(probs)

        val_loss = loss_sum / n
        y_true_va = np.concatenate(y_true_va)
        y_pred_va = np.concatenate(y_pred_va)
        val_acc = accuracy_score(y_true_va, y_pred_va)

        val_losses.append(val_loss)
        val_accs.append(val_acc)
        val_probs.append(torch.cat(epoch_val_probs, dim=0))       # [N_val, 5]

        print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.3f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.3f}")

        # ---- early stopping ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    training_time = time.perf_counter() - start
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses, train_accs, val_accs, train_probs, val_probs, training_time

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, in_ch: int, n_classes: int = 5, p_drop: float = 0.5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(in_ch, 16, 7, stride=2, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Conv1d(16, 32, 7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )

        self.dropout = nn.Dropout(p_drop)
        self.classifier = nn.Linear(32, n_classes)   # <-- was 1

    def forward(self, x):
        x = self.features(x)
        x = self.dropout(x)
        return self.classifier(x)                    # <-- return [B, 5] logits

In [ ]:
Epochs = 50
lr = 1e-3
patience = 10

n_classes = 5
model = CNN1D(len(SIGNAL_ORDER), n_classes)

t0 = time.time()
model, trL, vaL, trA, vaA, trP, valP, _ = train_model_with_loaders_multiclass(
    model,
    train_loader,
    val_loader,
    n_classes=n_classes,
    epochs=Epochs,
    lr=lr,
    patience=patience,
    class_weights=None   # optional
)
t1 = time.time()

print("Final train acc:", trA[-1])
print("Final val acc:", vaA[-1])

Epoch 01 | train loss 1.6225 acc 0.198 | val loss 1.6093 acc 0.235
Epoch 02 | train loss 1.5965 acc 0.238 | val loss 1.6149 acc 0.222
Epoch 03 | train loss 1.5695 acc 0.231 | val loss 1.6209 acc 0.170
Epoch 04 | train loss 1.5808 acc 0.233 | val loss 1.6189 acc 0.170
Epoch 05 | train loss 1.5539 acc 0.288 | val loss 1.6146 acc 0.163
Epoch 06 | train loss 1.5613 acc 0.262 | val loss 1.6126 acc 0.190
Epoch 07 | train loss 1.5347 acc 0.309 | val loss 1.6098 acc 0.190
Epoch 08 | train loss 1.5389 acc 0.307 | val loss 1.6090 acc 0.190
Epoch 09 | train loss 1.5276 acc 0.311 | val loss 1.6081 acc 0.190
Epoch 10 | train loss 1.5261 acc 0.332 | val loss 1.6072 acc 0.209
Epoch 11 | train loss 1.5230 acc 0.354 | val loss 1.6080 acc 0.248
Epoch 12 | train loss 1.4973 acc 0.356 | val loss 1.6141 acc 0.229
Epoch 13 | train loss 1.4913 acc 0.366 | val loss 1.6171 acc 0.242
Epoch 14 | train loss 1.4845 acc 0.368 | val loss 1.6174 acc 0.242
Epoch 15 | train loss 1.4760 acc 0.365 | val loss 1.6231 acc 0

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def eval_loader_multiclass(model, loader, device=None, name="EVAL"):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    y_true_list, y_pred_list = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device).long().view(-1)    # [B]

            logits = model(xb)                    # [B, 5]
            preds = torch.argmax(logits, dim=1)   # [B]

            y_true_list.append(yb.cpu().numpy())
            y_pred_list.append(preds.cpu().numpy())

    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    print(f"{name} accuracy (window-level): {acc:.4f} | balanced acc: {bacc:.4f}")
    return y_true, y_pred, acc, bacc

In [ ]:
y_tr, yhat_tr, acc_tr, bacc_tr = eval_loader_multiclass(model, train_loader, name="TRAIN")
y_va, yhat_va, acc_va, bacc_va = eval_loader_multiclass(model, val_loader, name="VAL")
y_te, yhat_te, acc_te, bacc_te = eval_loader_multiclass(model, test_loader, name="TEST")

TRAIN accuracy (window-level): 0.4132 | balanced acc: 0.4122
VAL accuracy (window-level): 0.2288 | balanced acc: 0.2111
TEST accuracy (window-level): 0.3492 | balanced acc: 0.3256


In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def eval_participant_level(model, X_p, y_p, device=None, name="TEST"):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    P, W, C, L = X_p.shape
    y_true = y_p.astype(int)

    y_pred = []
    with torch.no_grad():
        for i in range(P):
            xb = torch.tensor(X_p[i], dtype=torch.float32).to(device)  # [W, C, L]
            logits = model(xb)                                         # [W, 5]
            mean_logits = logits.mean(dim=0)                           # [5]
            pred = int(mean_logits.argmax().item())
            y_pred.append(pred)

    y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    print(f"{name} participant-level acc: {acc:.4f} | balanced acc: {bacc:.4f}")
    return y_true, y_pred, acc, bacc

In [ ]:
eval_participant_level(model, meta["X_train_p"],  meta["y_train_p"],  name="TRAIN")
eval_participant_level(model, meta["X_test_p"], meta["y_test_p"], name="TEST")

TRAIN participant-level acc: 0.4375 | balanced acc: 0.4224
TEST participant-level acc: 0.3810 | balanced acc: 0.3500


(array([3, 2, 0, 2, 1, 1, 0, 4, 0, 1, 4, 3, 3, 0, 3, 4, 0, 2, 1, 2, 2]),
 array([1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 2, 2, 0, 1, 3, 0, 0, 2, 1, 2]),
 0.38095238095238093,
 0.35)

In [ ]:


FULL_SIGNAL_ORDER = ["ECG_mV", "Fingerpulse", "Cannula", "Thermopod"]

signal_sets = [
    ["ECG_mV"],
    ["Fingerpulse"],
    ["Cannula"],                          # 1 signal
    ["Thermopod"],                        # 1 signal
    #["Cannula", "Thermopod"],             # 2 signals
    ["ECG_mV", "Fingerpulse"],            # 2 signals
    ["ECG_mV", "Cannula"],            # 2 signals
    ["ECG_mV", "Thermopod"],            # 2 signals
    ["ECG_mV", "Fingerpulse","Thermopod"],   # 3 signals
    ["ECG_mV", "Fingerpulse", "Cannula"],   # 3 signals
    FULL_SIGNAL_ORDER,                    # 4 signals
]

seeds = [1, 2, 3, 4, 5]


results = []
for keep in signal_sets:
    for seed in seeds:
        m = run_one_experiment(
            X_full=X_nose,
            y=y_nose,
            full_order=FULL_SIGNAL_ORDER,
            keep_signals=keep,
            seed=seed,
            Epochs=50,
            lr=1e-3,
            patience=10,
            batch_size=64,
        )
        print(f"done: seed={seed}, signals={keep} -> acc={m['test_acc']:.3f}, auc={m['test_roc_auc']:.3f}")
        results.append(m)
